In [1]:
pip install pandas dnspython openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import dns.resolver
import os

# --- Configuration ---
# List of files from your image
files_to_process = [
    "filtered_contacts.xlsx",
    "filtered_contactsold.csv"
]

def is_valid_domain(email):
    """
    Checks if the domain of the email actually has mail servers (MX records).
    Returns True if domain is alive, False if dead.
    """
    if not isinstance(email, str) or '@' not in email:
        return False
        
    try:
        domain = email.split('@')[1].strip()
        
        # Check for MX Record (Mail Exchange)
        records = dns.resolver.resolve(domain, 'MX')
        return True
        
    except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN, dns.resolver.LifetimeTimeout):
        return False
    except Exception:
        return False

def process_file(file_path):
    print(f"\n{'='*40}")
    print(f"📂 Processing: {file_path}")
    
    # 1. Detect file type and load data
    if not os.path.exists(file_path):
        print(f"❌ Error: File '{file_path}' not found.")
        return

    try:
        if file_path.endswith('.xlsx'):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path)
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        return

    # 2. Check if 'Email' column exists
    # (Handling case sensitivity just in case, e.g., 'email' vs 'Email')
    email_col = None
    for col in df.columns:
        if col.strip().lower() == 'email':
            email_col = col
            break
            
    if not email_col:
        print(f"⚠️  Skipping: Could not find an 'Email' column in {file_path}")
        print(f"   Found columns: {list(df.columns)}")
        return

    print(f"   Found {len(df)} emails. Validating domains...")

    # 3. Apply the validation
    df['Domain_Status'] = df[email_col].apply(lambda x: is_valid_domain(x))

    # 4. Separate Good vs Bad
    valid_emails = df[df['Domain_Status'] == True].copy()
    dead_emails = df[df['Domain_Status'] == False].copy()

    # Drop the helper column for the clean output
    valid_emails = valid_emails.drop(columns=['Domain_Status'])

    # 5. Save Output
    # We will save as CSV regardless of input to keep it simple
    output_filename = "Verified_" + os.path.splitext(file_path)[0] + ".csv"
    valid_emails.to_csv(output_filename, index=False)

    print(f"✅ COMPLETE")
    print(f"   Original Count: {len(df)}")
    print(f"   Valid Kept:     {len(valid_emails)}")
    print(f"   Removed:        {len(dead_emails)}")
    print(f"   Saved to:       {output_filename}")

    if len(dead_emails) > 0:
        print("   Sample removed:", dead_emails[email_col].head(3).values)

# --- Main Execution Loop ---
for file in files_to_process:
    process_file(file)


📂 Processing: filtered_contacts.xlsx
   Found 1999 emails. Validating domains...
✅ COMPLETE
   Original Count: 1999
   Valid Kept:     1954
   Removed:        45
   Saved to:       Verified_filtered_contacts.csv
   Sample removed: ['amitha.k@secure-24.com' 'anjani.salian@in.nbssap.com'
 'ankur.beri@niit-tech.com']

📂 Processing: filtered_contactsold.csv
   Found 6795 emails. Validating domains...
✅ COMPLETE
   Original Count: 6795
   Valid Kept:     6203
   Removed:        592
   Saved to:       Verified_filtered_contactsold.csv
   Sample removed: ['saniya_hemt@zuaiapp.com' 'jobs@zluck.om'
 'career@vmhgroupsmr.com, vikas@vmhgroupsmr.com']
